# 03 · Infrações
Transforma os sete blocos de IUF da aba RESPOSTA (formato largo) em uma
tabela com uma linha por IUF (formato longo).

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Caminhos do projeto (relativos à pasta notebooks/)
bronze_dir = Path("../dados/bronze")
silver_dir = Path("../dados/prata")
source_file = bronze_dir / "Controle de alertas - CMF-SEAMA.xlsx"

In [ ]:
# Lê a RESPOSTA e mantém só as linhas aprovadas no notebook 02
df_raw = pd.read_excel(source_file, sheet_name="RESPOSTA")
df_raw["linha_excel"] = df_raw.index + 2

valid_rows = pd.read_parquet(silver_dir / "respostas.parquet", columns=["linha_excel"])
df_raw = df_raw[df_raw["linha_excel"].isin(valid_rows["linha_excel"])]
print(len(df_raw), "linhas válidas")

In [ ]:
# Tradução das colunas de cada bloco (o bloco 1 tem grafia diferente dos demais)
def block_columns(n):
    return {
        ("IUF1" if n == 1 else f"IUF_{n}"): "iuf",
        f"serie_{n}": "serie",
        f"Fiscalizacao_IUF{n}": "id_fiscalizacao",
        f"Enquadramento_IUF{n}": "enquadramento",
        ("Embargo_IUF1 (ha)" if n == 1 else f"Embargo_IUF{n}"): "embargo_ha",
        f"Multa_IUF{n}": "multa_reais",
    }

# Empilha os sete blocos
blocks = []
for n in range(1, 8):
    columns = {"ID alerta": "id_alerta", **block_columns(n)}
    block = df_raw[["linha_excel", *columns]].rename(columns=columns)
    block["bloco"] = n
    blocks.append(block)

df_long = pd.concat(blocks, ignore_index=True)

# Bloco sem número de IUF = não houve aquela infração. Antes de descartar,
# confere se algum bloco tem valor mas perdeu o número
has_values = df_long[["embargo_ha", "multa_reais"]].notna().any(axis=1)
print((df_long["iuf"].isna() & has_values).sum(), "blocos com valor mas sem número de IUF")

df_long = df_long.dropna(subset=["iuf"])
print(len(df_long), "IUFs empilhados")
print(df_long["bloco"].value_counts().sort_index())

In [ ]:
# Converte números, aceitando vírgula decimal (ex.: "8.770,87")
def to_number(series):
    text = series.astype("string").str.strip()
    has_comma = text.str.contains(",", regex=False).fillna(False)
    text = text.where(~has_comma,
                      text.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(text, errors="coerce")

for column in ["iuf", "id_fiscalizacao", "embargo_ha", "multa_reais"]:
    converted = to_number(df_long[column])
    lost = df_long[column].notna() & converted.isna()
    print(f"{column}: {lost.sum()} valores não converteram")
    df_long[column] = converted

df_long["id_alerta"] = df_long["id_alerta"].astype("int64")
df_long["iuf"] = df_long["iuf"].astype("Int64")
df_long["id_fiscalizacao"] = df_long["id_fiscalizacao"].astype("Int64")
df_long["serie"] = df_long["serie"].astype("string").str.strip().str.upper()

In [ ]:
# O mesmo IUF aparece em mais de uma linha?
key = ["serie", "iuf"]
is_duplicated = df_long.duplicated(subset=key, keep=False)
print(is_duplicated.sum(), "linhas com IUF repetido |",
      df_long.loc[is_duplicated, key].drop_duplicates().shape[0], "IUFs distintos envolvidos")

show_columns = ["id_alerta", "bloco", "serie", "iuf", "id_fiscalizacao", "embargo_ha", "multa_reais"]
print(df_long.loc[is_duplicated, show_columns].sort_values(key).head(10).to_string())

In [ ]:
# 1. As cópias de um mesmo IUF são idênticas?
value_columns = ["id_fiscalizacao", "enquadramento", "embargo_ha", "multa_reais"]
n_versions = df_long.groupby(key)[value_columns].nunique(dropna=False)
conflicts = n_versions[(n_versions > 1).any(axis=1)]
print(len(conflicts), "IUFs com cópias divergentes")
print(conflicts.head(10).to_string())

In [ ]:
# 2. Quanto a dupla contagem inflaria os totais
naive_sum = df_long[["multa_reais", "embargo_ha"]].sum()
dedup_sum = df_long.drop_duplicates(subset=key)[["multa_reais", "embargo_ha"]].sum()
print(pd.DataFrame({"soma_ingenua": naive_sum, "soma_sem_repeticao": dedup_sum}).to_string())

In [ ]:
# Detalhe dos IUFs em conflito
conflict_rows = df_long.set_index(key).loc[conflicts.index].reset_index()
print(conflict_rows[show_columns].sort_values(key).to_string())

In [ ]:
# Infrações únicas: chave = série + número + fiscalização
key = ["serie", "iuf", "id_fiscalizacao"]

# Conflitos que a chave não resolve: fica o menor valor (critério conservador)
n_versions = (df_long.groupby(key, dropna=False)[["enquadramento", "embargo_ha", "multa_reais"]]
              .nunique(dropna=False))
remaining = n_versions[(n_versions > 1).any(axis=1)]
print(len(remaining), "IUFs com conflito resolvido pela regra do menor valor")
print(remaining.to_string())

df_infractions = (df_long.groupby(key, as_index=False, dropna=False)
                  .agg(enquadramento=("enquadramento", "first"),
                       embargo_ha=("embargo_ha", "min"),
                       multa_reais=("multa_reais", "min"),
                       n_alertas=("id_alerta", "nunique")))
df_infractions["enquadramento"] = df_infractions["enquadramento"].astype("string")

print(len(df_infractions), "infrações únicas")
print("Multas: R$", round(df_infractions["multa_reais"].sum(), 2))
print("Embargos:", round(df_infractions["embargo_ha"].sum(), 2), "ha")
print("Infrações que cobrem mais de um alerta:", (df_infractions["n_alertas"] > 1).sum())

In [ ]:
# Ponte alerta × fiscalização (das respostas e dos blocos de IUF)
df_responses = pd.read_parquet(silver_dir / "respostas.parquet")
df_bridge = (pd.concat([df_responses[["id_alerta", "id_fiscalizacao"]],
                        df_long[["id_alerta", "id_fiscalizacao"]]])
             .dropna()
             .drop_duplicates())

# Checagem bloqueante e salvamento
assert not df_infractions.duplicated(subset=key).any(), "Infração repetida após a deduplicação"
df_infractions.to_parquet(silver_dir / "infracoes.parquet", index=False)
df_bridge.to_parquet(silver_dir / "ponte_alerta_fiscalizacao.parquet", index=False)
print(len(df_bridge), "ligações alerta × fiscalização salvas")

In [ ]:
# Data de cada fiscalização: a data registrada na linha onde o laudo aparece
row_dates = df_responses[["linha_excel", "data_fiscalizacao"]]
block_dates = (df_long[["linha_excel", "id_fiscalizacao"]]
               .merge(row_dates, on="linha_excel", how="left", validate="many_to_one"))

df_inspection_dates = (pd.concat([df_responses[["id_fiscalizacao", "data_fiscalizacao"]],
                                  block_dates[["id_fiscalizacao", "data_fiscalizacao"]]])
                       .dropna(subset=["id_fiscalizacao"])
                       .groupby("id_fiscalizacao", as_index=False)["data_fiscalizacao"].min())

df_inspection_dates.to_parquet(silver_dir / "datas_fiscalizacao.parquet", index=False)
print(len(df_inspection_dates), "fiscalizações com data levantada |",
      df_inspection_dates["data_fiscalizacao"].isna().sum(), "sem data")